In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.base import TransformerMixin
from sklearn.pipeline import Pipeline

In [ ]:
df_amazon = pd.read_csv ("amazon_alexa.tsv", sep="\t",parse_dates=['date'],na_values=['?','NA'])

/tmp/ipykernel_1888/1278288662.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_amazon = pd.read_csv ("amazon_alexa.tsv", sep="\t",parse_dates=['date'],na_values=['?','NA'])


In [ ]:
df_amazon.head()

,rating,date,variation,verified_reviews,feedback
0,5,2018-07-31,Charcoal Fabric,Love my Echo!,1
1,5,2018-07-31,Charcoal Fabric,Loved it!,1
2,4,2018-07-31,Walnut Finish,"Sometimes while playing a game, you can answer...",1
3,5,2018-07-31,Charcoal Fabric,I have had a lot of fun with this thing. My 4 ...,1
4,5,2018-07-31,Charcoal Fabric,Music,1


In [ ]:
df_amazon.columns

Index(['rating', 'date', 'variation', 'verified_reviews', 'feedback'], dtype='object')

In [ ]:
df_amazon.shape

(3150, 5)

In [ ]:
df_amazon.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3150 entries, 0 to 3149
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   rating            3150 non-null   int64         
 1   date              3150 non-null   datetime64[ns]
 2   variation         3150 non-null   object        
 3   verified_reviews  3149 non-null   object        
 4   feedback          3150 non-null   int64         
dtypes: datetime64[ns](1), int64(2), object(2)
memory usage: 123.2+ KB


In [ ]:
df_amazon.feedback.value_counts()

,count
feedback,
1,2893
0,257


In [ ]:
import string , spacy
from spacy.lang.en.stop_words import STOP_WORDS


# Create our list of punctuation marks
punctuations = string.punctuation

# Create our list of stopwords
nlp = spacy.load('en_core_web_sm')
stop_words = STOP_WORDS

# Creating our tokenizer function
def spacy_tokenizer(sentence):

    doc = nlp(sentence)

    # Lemmatizing each token, converting to lowercase, and removing stop words, punctuation, spaces, and numbers
    mytokens = [
        token.lemma_.lower().strip() for token in doc
        if not token.is_stop and not token.is_punct and not token.is_space and not token.like_num
    ]

    # Filter out any empty strings that might have resulted from stripping or weird lemmatization
    mytokens = [token for token in mytokens if token]

    # return preprocessed list of tokens
    return mytokens

# Test the tokenizer to ensure it produces tokens
print(spacy_tokenizer('Hello World! This is a test sentence with numbers like 123 and some punctuation.'))

['hello', 'world', 'test', 'sentence', 'number', 'like', 'punctuation']


In [ ]:
# Custom transformer using spaCy
class predictors(TransformerMixin):
    def transform(self, X, **transform_params):
        # Cleaning Text
        return [clean_text(text) for text in X]

    def fit(self, X, y=None, **fit_params):
        return self

    def get_params(self, deep=True):
        return {}

# Basic function to clean the text
def clean_text(text):
    # Removing spaces and converting text into lowercase
    return text.strip().lower()

In [ ]:
bow_vector = CountVectorizer(tokenizer = spacy_tokenizer, ngram_range=(1,1))

In [ ]:
tfidf_vector = TfidfVectorizer(tokenizer = spacy_tokenizer)

In [ ]:
from sklearn.model_selection import train_test_split

X = df_amazon['verified_reviews'].astype("str") # the features we want to analyze
ylabels = df_amazon['feedback'] # the labels, or answers, we want to test against

X_train, X_test, y_train, y_test = train_test_split(X, ylabels, test_size=0.3)

In [ ]:
from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression()

# Create pipeline using Bag of Words
pipe = Pipeline([("cleaner", predictors()),
                 ('vectorizer', bow_vector),
                 ('classifier', classifier)])

# model generation
pipe.fit(X_train,y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Pipeline(steps=[('cleaner', <__main__.predictors object at 0x78f702d99fd0>),
                ('vectorizer',
                 CountVectorizer(tokenizer=<function spacy_tokenizer at 0x78f701fb23e0>)),
                ('classifier', LogisticRegression())])

In [ ]:
from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression()

# Create pipeline using Bag of Words
pipe = Pipeline([
                 ('vectorizer', tfidf_vector),
                 ('classifier', classifier)])

# model generation
pipe.fit(X_train,y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Pipeline(steps=[('vectorizer',
                 TfidfVectorizer(tokenizer=<function spacy_tokenizer at 0x78f701fb23e0>)),
                ('classifier', LogisticRegression())])

In [ ]:
from sklearn.preprocessing import FunctionTransformer

def clean_texts(X):
    return [text.strip().lower() for text in X]

cleaner = FunctionTransformer(clean_texts)
from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression()

# Create pipeline using Bag of Words
pipe = Pipeline([("cleaner", cleaner),
                 ('vectorizer', bow_vector),
                 ('classifier', classifier)])

# model generation
pipe.fit(X_train,y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Pipeline(steps=[('cleaner',
                 FunctionTransformer(func=<function clean_texts at 0x78f6fbacf240>)),
                ('vectorizer',
                 CountVectorizer(tokenizer=<function spacy_tokenizer at 0x78f701fb23e0>)),
                ('classifier', LogisticRegression())])

In [ ]:
from sklearn import metrics
# Predicting with a test dataset
predicted = pipe.predict(X_test)

# Model Accuracy
print("Logistic Regression Accuracy:",metrics.accuracy_score(y_test, predicted))
print("Logistic Regression Precision:",metrics.precision_score(y_test, predicted))
print("Logistic Regression Recall:",metrics.recall_score(y_test, predicted))
confusion_matrix = metrics.confusion_matrix(y_test, predicted)
print(confusion_matrix)

Logistic Regression Accuracy: 0.9396825396825397
Logistic Regression Precision: 0.9473684210526315
Logistic Regression Recall: 0.9896907216494846
[[ 24  48]
 [  9 864]]
